In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = "root"
DB_PASSWORD = "sjbarad777"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "paynova_fraud_analytics"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("SQLAlchemy connection created successfully.")

SQLAlchemy connection created successfully.


In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE();"))
    print(result.fetchone())

('paynova_fraud_analytics',)


In [4]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_transaction_value,
    ROUND(AVG(amount_inr), 2) AS average_transaction_value
FROM fact_transactions;
"""

result = pd.read_sql(text(query), engine)

result

,total_transactions,total_transaction_value,average_transaction_value
0,0,None,None


In [5]:
query = """
SELECT
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr), 2) AS total_transaction_value,
    ROUND(AVG(amount_inr), 2) AS average_transaction_value,
    ROUND(MIN(amount_inr), 2) AS minimum_transaction_value,
    ROUND(MAX(amount_inr), 2) AS maximum_transaction_value
FROM fact_transactions;
"""

kpi = pd.read_sql(text(query), engine)

kpi

,total_transactions,total_transaction_value,average_transaction_value,minimum_transaction_value,maximum_transaction_value
0,0,None,None,None,None


In [6]:
# success and failure rate by hour
query = """
SELECT
    transaction_status,
    COUNT(*) AS transactions,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM fact_transactions
GROUP BY transaction_status
ORDER BY transactions DESC;
"""

status_analysis = pd.read_sql(text(query), engine)

status_analysis

,transaction_status,transactions,percentage


In [7]:
query = """
SELECT
    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN transaction_status = 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS successful_transactions,

    SUM(
        CASE
            WHEN transaction_status = 'FAILED'
            THEN 1
            ELSE 0
        END
    ) AS failed_transactions,

    ROUND(
        SUM(
            CASE
                WHEN transaction_status = 'SUCCESS'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS success_rate_pct,

    ROUND(
        SUM(
            CASE
                WHEN transaction_status = 'FAILED'
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS failure_rate_pct

FROM fact_transactions;
"""

status_kpi = pd.read_sql(text(query), engine)

status_kpi

,total_transactions,successful_transactions,failed_transactions,success_rate_pct,failure_rate_pct
0,0,None,None,None,None


In [5]:
query = """
SELECT
    DATE_FORMAT(`transaction_timestamp`, '%Y-%m') AS transaction_month,
    COUNT(*) AS transactions,
    ROUND(SUM(amount_inr), 2) AS transaction_value,
    ROUND(AVG(amount_inr), 2) AS average_transaction_value
FROM fact_transactions
GROUP BY DATE_FORMAT(`transaction_timestamp`, '%Y-%m')
ORDER BY DATE_FORMAT(`transaction_timestamp`, '%Y-%m');
"""

monthly = pd.read_sql(text(query), engine)

monthly

,transaction_month,transactions,transaction_value,average_transaction_value


In [4]:
query = """
WITH monthly AS (

    SELECT
        DATE_FORMAT(`transaction_timestamp`, '%Y-%m') AS transaction_month,
        COUNT(*) AS transactions,
        SUM(amount_inr) AS transaction_value

    FROM fact_transactions

    GROUP BY DATE_FORMAT(`transaction_timestamp`, '%Y-%m')
)

SELECT
    transaction_month,
    transactions,

    ROUND(transaction_value, 2) AS transaction_value,

    LAG(transaction_value) OVER (
        ORDER BY transaction_month
    ) AS previous_month_value,

    ROUND(
        (
            transaction_value -
            LAG(transaction_value) OVER (
                ORDER BY transaction_month
            )
        ) * 100.0
        /
        NULLIF(
            LAG(transaction_value) OVER (
                ORDER BY transaction_month
            ),
            0
        ),
        2
    ) AS mom_growth_pct

FROM monthly

ORDER BY transaction_month;
"""

monthly_growth = pd.read_sql(text(query), engine)

monthly_growth

,transaction_month,transactions,transaction_value,previous_month_value,mom_growth_pct


In [6]:
# frud couitribution
query = """
SELECT
    merchant_category,

    SUM(fraud_flag) AS fraud_transactions,

    ROUND(
        SUM(fraud_flag) * 100.0 /
        SUM(SUM(fraud_flag)) OVER (),
        2
    ) AS fraud_contribution_pct

FROM fact_transactions

GROUP BY merchant_category

ORDER BY fraud_transactions DESC;
"""

fraud_contribution = pd.read_sql(text(query), engine)

fraud_contribution

,merchant_category,fraud_transactions,fraud_contribution_pct


In [7]:
# Risk category analysis
query = """
SELECT
    risk_category,

    COUNT(*) AS transactions,

    SUM(fraud_flag) AS fraud_transactions,

    ROUND(
        SUM(fraud_flag) * 100.0 / COUNT(*),
        3
    ) AS fraud_rate_pct,

    ROUND(
        SUM(amount_inr),
        2
    ) AS transaction_value

FROM fact_transactions

GROUP BY risk_category

ORDER BY fraud_rate_pct DESC;
"""

risk_analysis = pd.read_sql(text(query), engine)

risk_analysis

,risk_category,transactions,fraud_transactions,fraud_rate_pct,transaction_value


In [12]:
# high risk merchants(trasactions)
query = """
SELECT
    transaction_id,
    transaction_timestamp,
    amount_inr,
    transaction_type,
    merchant_category,
    sender_state,
    sender_bank,
    device_type,
    network_type,
    risk_score,
    risk_category,
    fraud_flag

FROM fact_transactions

WHERE risk_category = 'High'

ORDER BY risk_score DESC,
         amount_inr DESC

LIMIT 100;
"""

high_risk = pd.read_sql(text(query), engine)

high_risk

,transaction_id,transaction_timestamp,amount_inr,transaction_type,merchant_category,sender_state,sender_bank,device_type,network_type,risk_score,risk_category,fraud_flag


In [13]:
# daily frusd analysis
query = """
SELECT
    DATE(`transaction_timestamp`) AS transaction_date,

    COUNT(*) AS transactions,

    SUM(fraud_flag) AS fraud_transactions,

    ROUND(
        SUM(fraud_flag) * 100.0 / COUNT(*),
        3
    ) AS fraud_rate_pct

FROM fact_transactions

GROUP BY DATE(`transaction_timestamp`)

ORDER BY transaction_date;
"""

daily_fraud = pd.read_sql(text(query), engine)

daily_fraud.head()

,transaction_date,transactions,fraud_transactions,fraud_rate_pct


In [22]:
query = """

WITH daily AS (

    SELECT
        DATE(`transaction_timestamp`) AS transaction_date,

        COUNT(*) AS total_transactions,

        SUM(fraud_flag) AS fraud_transactions

    FROM fact_transactions

    GROUP BY DATE(`transaction_timestamp`)
),

rolling AS (

    SELECT
        transaction_date,
        total_transactions,
        fraud_transactions,

        SUM(total_transactions) OVER (
            ORDER BY transaction_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_transactions,

        SUM(fraud_transactions) OVER (
            ORDER BY transaction_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7d_fraud

    FROM daily
)

SELECT
    transaction_date,

    total_transactions,

    fraud_transactions,

    rolling_7d_transactions,

    rolling_7d_fraud,

    ROUND(
        rolling_7d_fraud * 100.0 /
        NULLIF(rolling_7d_transactions, 0),
        3
    ) AS rolling_7d_fraud_rate_pct

FROM rolling

ORDER BY transaction_date;

"""

rolling_7d_fraud = pd.read_sql(text(query), engine)

rolling_7d_fraud.head(10)

,transaction_date,total_transactions,fraud_transactions,rolling_7d_transactions,rolling_7d_fraud,rolling_7d_fraud_rate_pct


In [23]:
from pathlib import Path

output_dir = Path("../data/analytics")
output_dir.mkdir(parents=True, exist_ok=True)

fraud_state_query = """
SELECT
    sender_state,
    SUM(fraud_flag) AS fraud_transactions,
    ROUND(
        SUM(fraud_flag) * 100.0 / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM fact_transactions
GROUP BY sender_state
ORDER BY fraud_rate_pct DESC;
"""

fraud_state = pd.read_sql(text(fraud_state_query), engine)

monthly_growth.to_csv(
    output_dir / "monthly_growth.csv",
    index=False
)

fraud_contribution.to_csv(
    output_dir / "fraud_by_category.csv",
    index=False
)

fraud_state.to_csv(
    output_dir / "fraud_by_state.csv",
    index=False
)

risk_analysis.to_csv(
    output_dir / "risk_analysis.csv",
    index=False
)

print(f"Analytics exported to {output_dir.resolve()}")

Analytics exported to E:\my new for the me last for the change the my life for the life only goal\project\Digital Payments Fraud Intelligence\data\analytics
